# Standalone edition
This notebook is self-contained. It embeds the utility functions directly, so it can be uploaded as one `.ipynb` file without downloading the companion Python package. Install PyTorch if needed.

# 03 — AI Model Assurance & Counterfactual Optimization
### Product workflows using an architecture-aware decoder-only Transformer

This notebook trains a small synthetic next-token model, then demonstrates artifact intake, tamper evidence, attention-head ablation, candidate evaluation, and auditable policy gates. The model is intentionally small and randomly initialized; it is a test fixture, not a production language model.


In [1]:
import copy
import math
import numpy as np

import hashlib
import json
from pathlib import Path
from typing import Any

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

def state_dict_sha256(model: nn.Module) -> str:
    digest = hashlib.sha256()
    for name, tensor in sorted(model.state_dict().items()):
        t = tensor.detach().cpu().contiguous()
        digest.update(name.encode("utf-8"))
        digest.update(str(t.dtype).encode("ascii"))
        digest.update(str(tuple(t.shape)).encode("ascii"))
        digest.update(t.reshape(-1).view(torch.uint8).numpy().tobytes())
    return digest.hexdigest()

def tensor_inventory(model: nn.Module) -> list[dict[str, Any]]:
    rows = []
    for name, tensor in model.state_dict().items():
        t = tensor.detach().float().cpu()
        rows.append({
            "name": name, "shape": list(tensor.shape), "dtype": str(tensor.dtype),
            "numel": int(tensor.numel()),
            "mean": float(t.mean()) if t.numel() else 0.0,
            "std": float(t.std(unbiased=False)) if t.numel() else 0.0,
            "min": float(t.min()) if t.numel() else 0.0,
            "max": float(t.max()) if t.numel() else 0.0,
            "finite": bool(torch.isfinite(t).all()) if t.numel() else True,
            "zero_fraction": float((t == 0).float().mean()) if t.numel() else 0.0,
        })
    return rows

def architecture_summary(model: nn.Module) -> dict[str, Any]:
    return {
        "model_class": model.__class__.__name__,
        "modules": [{"name": n, "class": m.__class__.__name__}
                    for n, m in model.named_modules()],
        "parameter_count": int(sum(p.numel() for p in model.parameters())),
        "trainable_parameter_count": int(sum(
            p.numel() for p in model.parameters() if p.requires_grad)),
        "state_tensor_count": len(model.state_dict()),
    }

@torch.no_grad()
def compare_outputs(model_a, model_b, probe_x):
    old_a, old_b = model_a.training, model_b.training
    try:
        model_a.eval(); model_b.eval()
        a, b = model_a(probe_x), model_b(probe_x)
        if a.shape != b.shape:
            raise ValueError("Output shape mismatch")
        return {
            "max_abs_error": float((a-b).abs().max()),
            "mean_abs_error": float((a-b).abs().mean()),
            "prediction_agreement": float(
                (a.argmax(-1) == b.argmax(-1)).float().mean())
        }
    finally:
        model_a.train(old_a); model_b.train(old_b)

def gradient_sensitivity(model, x, y):
    was_training = model.training
    model.zero_grad(set_to_none=True)
    try:
        model.train()
        loss = F.cross_entropy(model(x), y)
        loss.backward()
        values = {}
        for name, p in model.named_parameters():
            if p.grad is None:
                values[name] = None
            else:
                g = p.grad.detach().abs()
                values[name] = {"mean_abs_grad": float(g.mean()),
                                "max_abs_grad": float(g.max())}
        return {"loss": float(loss.detach()), "parameters": values}
    finally:
        model.zero_grad(set_to_none=True)
        model.train(was_training)

def scan_model_tensors(model, max_abs_threshold=1e3):
    inventory = tensor_inventory(model)
    findings = []
    for row in inventory:
        if not row["finite"]:
            findings.append({"severity":"high","code":"NON_FINITE_TENSOR",
                             "tensor":row["name"]})
        if row["numel"] and row["zero_fraction"] == 1.0:
            findings.append({"severity":"info","code":"ALL_ZERO_TENSOR",
                             "tensor":row["name"]})
        if row["numel"] and max(abs(row["min"]), abs(row["max"])) > max_abs_threshold:
            findings.append({"severity":"medium","code":"EXTREME_MAGNITUDE",
                             "tensor":row["name"]})
    return {
        "status": "findings" if findings else "no_heuristic_findings",
        "findings": findings,
        "disclaimer": "Heuristic checks are triage signals, not a safety verdict."
    }

def make_manifest(model, model_name, source, license_name=None):
    return {
        "schema_version":"1.0", "model_name":model_name, "source":source,
        "license":license_name, "state_dict_sha256":state_dict_sha256(model),
        "architecture":architecture_summary(model),
        "tensor_inventory":tensor_inventory(model)
    }

def save_json(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2), encoding="utf-8")
    return path

class WeightRowTokenizer:
    """Prototype numeric matrix-row encoder; not a trained Weight Oracle."""
    def __init__(self, max_rows=256, max_cols=256):
        self.max_rows, self.max_cols = max_rows, max_cols
    def encode(self, name, matrix):
        if matrix.ndim != 2:
            raise ValueError("Expected a 2-D tensor")
        x = matrix.detach().float().cpu()[:self.max_rows, :self.max_cols]
        mean = x.mean(1, keepdim=True)
        std = x.std(1, keepdim=True, unbiased=False).clamp_min(1e-8)
        stats = torch.stack([mean[:,0], std[:,0], x.abs().mean(1), x.norm(dim=1)], dim=1)
        return {"tensor_name":name, "values":(x-mean)/std,
                "row_statistics":stats, "original_shape":list(matrix.shape),
                "truncated_shape":list(x.shape)}


In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 41
torch.manual_seed(SEED)
# Avoid CPU thread oversubscription in this small demonstration model.
if DEVICE == 'cpu':
    torch.set_num_threads(1)
print("Standalone notebook | device:", DEVICE)

Standalone notebook | device: cpu


## 1. Reproducible synthetic next-token task and baseline

We generate short token sequences from a simple repeating grammar. Inputs and next-token targets are separated into train, validation, and test splits. Candidate selection uses validation data; test data is reserved for final reporting.


In [3]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps
    def forward(self, x):
        scale = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return x * scale * self.weight

class CausalSelfAttention(nn.Module):
    def __init__(self, dim=128, heads=4):
        super().__init__()
        assert dim % heads == 0
        self.dim, self.heads, self.head_dim = dim, heads, dim // heads
        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)
        self.o_proj = nn.Linear(dim, dim, bias=False)
    def forward(self, x):
        b, t, d = x.shape
        def split_heads(z):
            return z.view(b, t, self.heads, self.head_dim).transpose(1, 2)
        q, k, v = map(split_heads, (self.q_proj(x), self.k_proj(x), self.v_proj(x)))
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        causal = torch.triu(torch.ones(t, t, device=x.device, dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(causal, torch.finfo(scores.dtype).min)
        weights = torch.softmax(scores, dim=-1)
        context = weights @ v
        context = context.transpose(1, 2).contiguous().view(b, t, d)
        return self.o_proj(context)

class SwiGLU(nn.Module):
    def __init__(self, dim=128, hidden_dim=256):
        super().__init__()
        self.gate_proj = nn.Linear(dim, hidden_dim, bias=False)
        self.up_proj = nn.Linear(dim, hidden_dim, bias=False)
        self.down_proj = nn.Linear(hidden_dim, dim, bias=False)
    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

class TransformerBlock(nn.Module):
    def __init__(self, dim=128, heads=4, mlp_dim=256):
        super().__init__()
        self.attn_norm = RMSNorm(dim)
        self.attn = CausalSelfAttention(dim, heads)
        self.ffn_norm = RMSNorm(dim)
        self.ffn = SwiGLU(dim, mlp_dim)
    def forward(self, x):
        x = x + self.attn(self.attn_norm(x))
        x = x + self.ffn(self.ffn_norm(x))
        return x

class MiniDecoderTransformer(nn.Module):
    """Small decoder-only LM for architecture-aware experiments; not pretrained."""
    def __init__(self, vocab_size=32, max_seq_len=16, dim=128, heads=4,
                 layers=2, mlp_dim=256, tie_embeddings=True):
        super().__init__()
        self.config = dict(vocab_size=vocab_size, max_seq_len=max_seq_len,
                          dim=dim, heads=heads, layers=layers, mlp_dim=mlp_dim,
                          tie_embeddings=tie_embeddings)
        self.token_embedding = nn.Embedding(vocab_size, dim)
        self.position_embedding = nn.Embedding(max_seq_len, dim)
        self.blocks = nn.ModuleList([
            TransformerBlock(dim, heads, mlp_dim) for _ in range(layers)
        ])
        self.final_norm = RMSNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)
        if tie_embeddings:
            self.lm_head.weight = self.token_embedding.weight
    def forward(self, input_ids):
        b, t = input_ids.shape
        if t > self.config['max_seq_len']:
            raise ValueError('Sequence exceeds configured max_seq_len')
        positions = torch.arange(t, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)[None, :, :]
        for block in self.blocks:
            x = block(x)
        return self.lm_head(self.final_norm(x))

def permute_attention_heads_(model, layer_index, permutation):
    """Function-preserving head permutation for this explicit MHA implementation."""
    attn = model.blocks[layer_index].attn
    p = torch.as_tensor(permutation, dtype=torch.long)
    if sorted(p.tolist()) != list(range(attn.heads)):
        raise ValueError('permutation must contain each attention head exactly once')
    idx = torch.cat([torch.arange(int(h)*attn.head_dim, (int(h)+1)*attn.head_dim) for h in p]).to(attn.q_proj.weight.device)
    with torch.no_grad():
        for proj in (attn.q_proj, attn.k_proj, attn.v_proj):
            proj.weight.copy_(proj.weight[idx, :])
        attn.o_proj.weight.copy_(attn.o_proj.weight[:, idx])
    return model

def permute_swiglu_units_(model, layer_index, permutation):
    """Function-preserving permutation of the paired SwiGLU intermediate units."""
    ffn = model.blocks[layer_index].ffn
    p = torch.as_tensor(permutation, dtype=torch.long)
    if sorted(p.tolist()) != list(range(ffn.gate_proj.out_features)):
        raise ValueError('permutation must contain every SwiGLU intermediate unit')
    with torch.no_grad():
        ffn.gate_proj.weight.copy_(ffn.gate_proj.weight[p])
        ffn.up_proj.weight.copy_(ffn.up_proj.weight[p])
        ffn.down_proj.weight.copy_(ffn.down_proj.weight[:, p])
    return model

@torch.no_grad()
def ablate_attention_head(model, layer_index, head_index):
    """Ablate one head's contribution by zeroing its input slice to O projection."""
    candidate = copy.deepcopy(model)
    attn = candidate.blocks[layer_index].attn
    if not 0 <= head_index < attn.heads:
        raise IndexError('head_index out of range')
    start, end = head_index * attn.head_dim, (head_index + 1) * attn.head_dim
    attn.o_proj.weight[:, start:end].zero_()
    return candidate

@torch.no_grad()
def ablate_swiglu_unit(model, layer_index, unit_index):
    candidate = copy.deepcopy(model)
    ffn = candidate.blocks[layer_index].ffn
    if not 0 <= unit_index < ffn.gate_proj.out_features:
        raise IndexError('unit_index out of range')
    ffn.gate_proj.weight[unit_index].zero_()
    ffn.up_proj.weight[unit_index].zero_()
    ffn.down_proj.weight[:, unit_index].zero_()
    return candidate

def next_token_loss(model, input_ids, target_ids):
    logits = model(input_ids)
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)), target_ids.reshape(-1))

def make_sequences(n, seq_len=12, vocab_size=32, seed=1):
    g = torch.Generator().manual_seed(seed)
    # A simple deterministic grammar with controlled noise: token[t+1] follows token[t] + 1 mod vocab.
    starts = torch.randint(0, vocab_size, (n, 1), generator=g)
    offsets = torch.arange(seq_len + 1).view(1, -1)
    seq = (starts + offsets) % vocab_size
    noise_mask = torch.rand(n, seq_len + 1, generator=g) < 0.04
    noise = torch.randint(0, vocab_size, (n, seq_len + 1), generator=g)
    seq = torch.where(noise_mask, noise, seq)
    return seq[:, :-1].long(), seq[:, 1:].long()

x_train, y_train = make_sequences(1800, seed=1)
x_val, y_val = make_sequences(400, seed=2)
x_test, y_test = make_sequences(500, seed=3)
train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=64, shuffle=True)

# Architecture defined in this notebook: 2 layers, width 128, 4 heads, SwiGLU, RMSNorm.
# The shared architecture helpers are embedded below so this notebook is independently uploadable.

def train(model, loader, epochs=3):
    opt = torch.optim.AdamW(model.parameters(), lr=0.002)
    model.train()
    for epoch in range(epochs):
        total = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), yb.reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item()
        print(f'Epoch {epoch+1}: mean batch loss={total/len(loader):.4f}')
    return model

def evaluate(model, x, y):
    model.eval()
    with torch.no_grad():
        logits = model(x.to(DEVICE))
        yy = y.to(DEVICE)
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), yy.reshape(-1))
        accuracy = (logits.argmax(-1) == yy).float().mean()
        return {'loss': float(loss), 'token_accuracy': float(accuracy)}

baseline = train(MiniDecoderTransformer().to(DEVICE), train_loader)
baseline_val = evaluate(baseline, x_val, y_val)
baseline_test = evaluate(baseline, x_test, y_test)
print('Validation:', baseline_val, '| Test:', baseline_test)


Epoch 1: mean batch loss=17.8005


Epoch 2: mean batch loss=0.6275


Epoch 3: mean batch loss=0.4937


Validation: {'loss': 0.5224583148956299, 'token_accuracy': 0.9160416722297668} | Test: {'loss': 0.49824434518814087, 'token_accuracy': 0.9213333129882812}


## 2. Use case A — model supply-chain intake

A model artifact needs more than a filename. Capture source, license, architecture, tensor inventory, hash, and basic integrity checks. In production, add signed attestations, trusted registry identity, provenance verification, and a controlled artifact loader.


In [4]:
intake_manifest = make_manifest(
    baseline, model_name='mini-decoder-transformer-demo',
    source='local-synthetic-training-run', license_name='synthetic-demo'
)
intake_scan = scan_model_tensors(baseline)
print('Artifact hash:', intake_manifest['state_dict_sha256'])
print('Architecture:', intake_manifest['architecture']['model_class'])
print('Structural scan:', intake_scan['status'])

Artifact hash: 24a14eaa8e45f4fd2b9358287b258427271c6b62cf63451aceaa1c33b1a95c51
Architecture: MiniDecoderTransformer
Structural scan: no_heuristic_findings


### Demonstrate tamper evidence

Change one query-projection parameter in a copy. The hash should change. A hash mismatch proves the bytes differ from the recorded baseline; it does not tell us whether the change is malicious or behaviorally significant.


In [5]:
tampered = copy.deepcopy(baseline)
with torch.no_grad():
    tampered.blocks[0].attn.q_proj.weight[0, 0] += 0.25
print('Baseline hash:', state_dict_sha256(baseline))
print('Changed hash: ', state_dict_sha256(tampered))
assert state_dict_sha256(baseline) != state_dict_sha256(tampered)

Baseline hash: 24a14eaa8e45f4fd2b9358287b258427271c6b62cf63451aceaa1c33b1a95c51
Changed hash:  883d2e564c7e5db785da990685ecff19b8faef18cdfea92a85485e7cada8b206


## 3. Use case B — model change assurance

Compare a candidate against the baseline on fixed token probes and task-level validation. Aggregate metrics can conceal localized regressions, so production evaluations should include long-context, multilingual, adversarial, safety, and customer-specific task suites as appropriate.


In [6]:
probe = x_val[:128].to(DEVICE)
functional_delta = compare_outputs(baseline, tampered, probe)
tampered_val = evaluate(tampered, x_val, y_val)
print('Functional comparison:', functional_delta)
print('Baseline validation:', baseline_val)
print('Tampered validation:', tampered_val)

Functional comparison: {'max_abs_error': 0.036536216735839844, 'mean_abs_error': 0.0006343162385746837, 'prediction_agreement': 1.0}
Baseline validation: {'loss': 0.5224583148956299, 'token_accuracy': 0.9160416722297668}
Tampered validation: {'loss': 0.5224685072898865, 'token_accuracy': 0.9160416722297668}


## 4. Use case C — controlled Transformer component ablation

Rank attention-head ablations in the first block using validation next-token loss. This is a negative-space analysis experiment: it identifies components with lower measured impact under this particular evaluation distribution. It does not establish global redundancy.


In [7]:
effects = []
for head in range(baseline.blocks[0].attn.heads):
    candidate_head = ablate_attention_head(baseline, layer_index=0, head_index=head)
    metrics = evaluate(candidate_head, x_val, y_val)
    effects.append({'layer': 0, 'attention_head': head,
                    'delta_loss': metrics['loss'] - baseline_val['loss'],
                    'delta_token_accuracy': metrics['token_accuracy'] - baseline_val['token_accuracy']})
ranked = sorted(effects, key=lambda row: row['delta_loss'])
ranked

[{'layer': 0,
  'attention_head': 1,
  'delta_loss': 0.004546463489532471,
  'delta_token_accuracy': 0.00020831823348999023},
 {'layer': 0,
  'attention_head': 2,
  'delta_loss': 0.020937323570251465,
  'delta_token_accuracy': -0.0006250143051147461},
 {'layer': 0,
  'attention_head': 0,
  'delta_loss': 0.022532999515533447,
  'delta_token_accuracy': -0.00041669607162475586},
 {'layer': 0,
  'attention_head': 3,
  'delta_loss': 0.023725509643554688,
  'delta_token_accuracy': -0.0006250143051147461}]

## 5. Explicit policy gate

A candidate is not promoted merely because its ablation score is low. Define measurable tolerances, run the required regression suite, and retain a human approval step. The example thresholds are demonstration values, not universal production defaults.


In [8]:
chosen_head = ranked[0]['attention_head']
candidate = ablate_attention_head(baseline, layer_index=0, head_index=chosen_head)
candidate_val = evaluate(candidate, x_val, y_val)

MAX_LOSS_INCREASE = 0.01
MAX_TOKEN_ACCURACY_DROP = 0.01
quality_gate = (
    candidate_val['loss'] <= baseline_val['loss'] + MAX_LOSS_INCREASE
    and candidate_val['token_accuracy'] >= baseline_val['token_accuracy'] - MAX_TOKEN_ACCURACY_DROP
)
security_gate = intake_scan['status'] == 'no_heuristic_findings'
human_approval_required = True
promotion_status = 'ELIGIBLE_FOR_REVIEW' if quality_gate and security_gate else 'BLOCKED'
print({'layer': 0, 'attention_head': chosen_head, 'quality_gate': quality_gate,
       'security_gate': security_gate, 'human_approval_required': human_approval_required,
       'promotion_status': promotion_status, 'candidate_validation': candidate_val})

{'layer': 0, 'attention_head': 1, 'quality_gate': True, 'security_gate': True, 'human_approval_required': True, 'promotion_status': 'ELIGIBLE_FOR_REVIEW', 'candidate_validation': {'loss': 0.5270047783851624, 'token_accuracy': 0.9162499904632568}}


## 6. Model-security test plan: what a real product must add

The synthetic checks above are only the first layer. A customer-facing model assurance product should support:

| Control | Evidence / test |
|---|---|
| Artifact integrity | SHA-256 plus signed provenance/attestation |
| Unsafe serialization | Prefer Safetensors; restrict untrusted pickle loading |
| Dependency supply chain | SBOM, pinned versions, vulnerability scan |
| Model provenance | Source registry, author, license, training/data declarations |
| Backdoor investigation | Trigger suites, activation analysis, causal interventions |
| Behavioral regression | Task suites, subgroup tests, adversarial prompts |
| Privacy leakage | Memorization and extraction evaluations |
| Runtime security | Container image scan, least privilege, network egress controls |
| Deployment governance | Approval, signed artifact, rollback, audit log |

Weight-space anomaly scores should be one signal among several, not a stand-alone backdoor detector.

## 7. Export an assurance report

The report is structured so a future FastAPI service or AI security dashboard can consume it. A production schema should be versioned and include dataset IDs, runtime versions, signatures, reviewer identity, and policy decisions.

In [9]:
# Test set is evaluated only after validation-based candidate selection.
candidate_test = evaluate(candidate, x_test, y_test)
report = {
    'schema': 'model-assurance-report-v1',
    'model': intake_manifest,
    'integrity_scan': intake_scan,
    'tamper_demo': {
        'baseline_hash': state_dict_sha256(baseline),
        'candidate_hash': state_dict_sha256(tampered),
        'hashes_differ': state_dict_sha256(baseline) != state_dict_sha256(tampered)
    },
    'optimization': {
        'method': 'first_block_attention_head_ablation',
        'layer': 0, 'attention_head': chosen_head,
        'baseline_validation': baseline_val, 'candidate_validation': candidate_val,
        'candidate_test': candidate_test, 'quality_gate': bool(quality_gate),
        'promotion_status': promotion_status, 'human_approval_required': True
    },
    'limitations': [
        'Small synthetic decoder-only Transformer demonstration; not a pretrained LLM.',
        'Heuristic tensor scans do not prove a model is safe or unsafe.',
        'Low measured impact on one validation distribution does not prove global irrelevance.',
        'Third-party Transformer architectures require verified architecture adapters and broader evaluation.'
    ]
}
report_path = Path('model_assurance_report.json')
save_json(report, report_path)
print('Wrote:', report_path)

Wrote: model_assurance_report.json


## 8. Product integration pattern

A future application can expose the same engine through a FastAPI service:

- `POST /models/intake` — accept a supported artifact reference and metadata.
- `GET /models/{id}/manifest` — retrieve architecture, hash, and provenance.
- `POST /models/{id}/assess` — run structural and behavioral checks.
- `POST /models/{id}/experiments` — run isolated counterfactual experiments.
- `GET /experiments/{id}/report` — retrieve evidence and policy results.
- `POST /models/{id}/promote` — require authorization and signed approval.

Keep the analysis service separate from the deployment controller. The engine recommends; policy and authorized operators decide.

## 9. Rego policy starter

The repository includes `policies/model_assurance.rego`. It demonstrates policy gates for required metadata, artifact hash presence, and an explicit review status. It is intentionally a starter policy and must be adapted to the exact report schema and threat model before production use.

## Exit criteria for a customer-facing proof of concept

- [ ] Model intake creates a reproducible manifest and hash.
- [ ] Structural scans report evidence and limitations.
- [ ] Candidate edits run in isolated copies.
- [ ] Behavioral regression uses versioned evaluation data.
- [ ] Security and quality gates are separate.
- [ ] Reports are machine-readable and auditable.
- [ ] Promotion requires human authorization and rollback capability.

**Commercial framing:** Model Assurance for AI applications — model supply-chain integrity, structural diagnostics, behavior regression, and controlled optimization. The research differentiator is weight-space-aware analysis; the product promise is measurable evidence and safer model lifecycle decisions.